In [20]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [21]:
# Tasks

"""
For a given input topic , first generate a detailed outline and then using topic name and detailed outline generate a detailed blog post. The output of the first task is used as input to the second task.

"""

load_dotenv()  # Load environment variables from .env file

True

In [22]:
chatModel = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7, max_tokens=300)
outputParser = StrOutputParser()

In [23]:
class Topic(TypedDict):
    topic: str
    outline: str
    blog_post: str

In [24]:
prompt1 = "Generate a detailed outline for a blog post on the topic: {topic}"
prompt2 = "Using the topic: {topic} and the outline: {outline}, generate a detailed blog post."

In [25]:
graph = StateGraph(Topic)

In [26]:
def generate_outline(input: Topic) -> Topic:
    topic = input["topic"]
    chain = chatModel | outputParser
    outline = chain.invoke(prompt1.format(topic=topic))
    input["outline"] = outline
    return input


def generate_blog_post(input: Topic) -> Topic:
    topic = input["topic"]
    outline = input["outline"]
    chain = chatModel | outputParser
    blog_post = chain.invoke(prompt2.format(topic=topic, outline=outline))
    input["blog_post"] = blog_post
    return input

In [27]:
graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_blog_post', generate_blog_post)

graph.add_edge('generate_outline', 'generate_blog_post')
graph.add_edge(START, 'generate_outline')
graph.add_edge('generate_blog_post', END)

llmWorkflow = graph.compile()

In [28]:
finalState = llmWorkflow.invoke({"topic": "The Future of Artificial Intelligence in Healthcare"})
print(finalState)

{'topic': 'The Future of Artificial Intelligence in Healthcare', 'outline': 'I. Introduction\n    A. Brief overview of the current state of artificial intelligence in healthcare\n    B. Importance of AI in revolutionizing the healthcare industry\n    C. Preview of the potential impact of AI on the future of healthcare\n\nII. Current Applications of AI in Healthcare\n    A. Diagnostic tools using machine learning algorithms\n    B. Personalized treatment plans based on patient data and AI analysis\n    C. Robot-assisted surgeries and other AI-driven procedures\n    D. Virtual health assistants for patient communication and support\n\nIII. Opportunities for AI in Healthcare\n    A. Predictive analytics for disease prevention and early intervention\n    B. Data-driven decision-making for healthcare providers\n    C. Automation of administrative tasks to improve efficiency and reduce costs\n    D. Integration of AI with wearable devices and telemedicine for remote patient monitoring\n\nIV.